# MEA analysis walkthrough

The general-purpose entry point for the MEA stack, and the front half of every per-protocol notebook. End to end: ingest experiment metadata into DataJoint, survey what's in there, find every date that ran a given protocol, judge which of those datasets is worth analyzing, build a pipeline for the one you pick, and check that the pieces line up before any analysis rests on them.

**Nothing here is specific to one stimulus.** Change the search string in §4 and every section works as written — §5's mosaics and chunk summaries, §6's pipeline, §6b's cluster-match QC, §9's rasters and §10's per-epoch counts all take a protocol block and say what is in it. What they deliberately do *not* do is interpret conditions: no section below knows what `currentBackgroundScale` or a drifting-grating mean is, because that is where protocols stop resembling each other.

**That interpretation belongs in a per-protocol notebook.** The intended shape is to copy this one, set the §4 search string, and continue past §10 with the condition axes that protocol actually has — grouping epochs by parameter, PSTHs by condition, whatever the experiment was asking. Everything up to §10 stays as-is and stays shared, so a fix here reaches every protocol.

The sections form two halves. **§1–§5 choose a dataset**: what's in the database, which dates ran the protocol, and which of those has clean enough typing to be worth the effort. **§6–§10 build and check one**: the pipeline, whether the noise chunk and the protocol datafile actually describe the same cells, and whether the block held together from first epoch to last.

## 1. Imports and setup

`retinanalysis` resolves its public API lazily, so this cell stays cheap — DataJoint is only pulled in when the first database-backed function is called, in §2.

In [ ]:
import retinanalysis as ra
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
import os

# Cell types used throughout. Defined here rather than in §5 because §6 and §7
# need them too, and §5 is optional — you skip it once you know which dataset
# you want, and a constant defined there would leave the rest NameError-ing.
#
#   cell_types — everything worth drawing a mosaic for.
#   MAIN_TYPES — the subset carried into the per-dataset views, where one
#                panel or line per type has to stay readable.
cell_types = ['OnP', 'OffP', 'OnM', 'OffM', 'A2', 'OnS', 'OffS']
MAIN_TYPES = ['OnP', 'OffP', 'OnM', 'OffM']

## 2. Populate the database

Ingests every experiment found on the configured source volumes into DataJoint, and re-ingests any date whose meta/tags `.json` has been modified since it was added (source mtime vs `Experiment.date_added`).

Volumes are swept in read-priority order — **ChrisNewSSD → ChrisProSSD → NAS** — and the first drive holding a given date wins, so a duplicate copy sitting on a slower volume never re-triggers an ingest. `ra.ingest_source_dirs()` reports the `(h5, meta, tags)` triples that will actually be searched; a drive that isn't mounted silently drops out of the list. The order itself lives in `src/retinanalysis/config/config.ini`, one section per volume.

Safe to re-run: dates already in the database and unchanged on disk are skipped.

In [3]:
# Volumes that will be swept, in read-priority order (local SSDs before the NAS).
for h5_dir, meta_dir, tags_dir in ra.ingest_source_dirs():
    print(f'h5   : {h5_dir}\nmeta : {meta_dir}\ntags : {tags_dir}\n')

# Ingest new dates, and refresh any date whose json changed since it was added.
# Returns {'n_ingested', 'added', 'updated', 'skipped'}.
summary = ra.populate_database()

print(f"\nnewly added : {len(summary['added'])}")
print(f"refreshed   : {len(summary['updated'])}")
print(f"errored     : {len(summary['skipped'])}")

h5   : /Volumes/ChrisProSSD/data/h5
meta : /Volumes/ChrisProSSD/data/datajoint_testbed/mea/meta
tags : /Volumes/ChrisProSSD/data/datajoint_testbed/mea/tags

Ingest source: /Volumes/ChrisProSSD/data/h5
Could not find data directory for 20220405C.json
Could not find data directory for 20220406C.json
Could not find data directory for 20220412C.json
Could not find data directory for 20220420C.json
Could not find data directory for 20220426C.json
Could not find data directory for 20220518C.json
Could not find data directory for 20220526C.json
Could not find data directory for 20220531C.json
Could not find data directory for 20220607C.json
Could not find data directory for 20220705C.json
Could not find data directory for 20220712C.json
Could not find data directory for 20220726C.json
Could not find data directory for 20220809C.json
Could not find data directory for 20220816C.json
Could not find data directory for 20220818C.json
Could not find data directory for 20220829C.json
Could not find 

Experiments:   0%|          | 0/30 [00:00<?, ?it/s]

Already in database: 20260113C
Already in database: 20251112C
Already in database: 20251215C
Already in database: 20251022C
Already in database: 20260102C
Already in database: 20230111C
Already in database: 20251008C
Already in database: 20250121C
Already in database: 20251006C
Already in database: 20230214C
Already in database: 20250924C
Already in database: 20250514C
Already in database: 20230228C
Already in database: 20250429C
Already in database: 20250321C
Already in database: 20230313C
Already in database: 20250306C
Already in database: 20230502C
Already in database: 20230516C
Already in database: 20230725C
Already in database: 20231026C
Already in database: 20231220C
Already in database: 20240117C
Already in database: 20240130C
Already in database: 20240523C
Already in database: 20240801C
Already in database: 20220823C
Already in database: 20221101C
Already in database: 20221123C
Already in database: 20260318C

No experiments skipped due to errors.
No new sorting chunks found in 

## 3. What's in the database?

A read-only survey of everything §2 ingested, before narrowing to one protocol. The implementations live in `retinanalysis/utils/db_summary.py` so this section stays two calls.

The first cell answers three questions:

1. **How many recordings of each kind** — `ra.recording_counts()`, split by `Experiment.is_mea` into MEA arrays and single-cell patch experiments.
2. **Which species** — `ra.species_counts()`, from `Animal.species`. Recorded for most MEA dates but rarely for patch experiments, so expect a large `(not recorded)` row on the patch side; that means the field is blank, not that the animal is missing.
3. **Which protocols dominate the MEA corpus** — `ra.mea_protocol_counts()`, ranked by number of distinct dates rather than number of blocks. Dates is the more useful denominator: a protocol run four times in one day is still one day of data. Both counts are returned so the difference is visible.

The protocol table is rendered with `ra.scroll_table` rather than `display()`, so the **whole** list is there — a fixed-height box with a sticky header, scrolling only once the corpus outgrows it. It used to be truncated to `head(15)`, which hid exactly the long tail you go looking for when hunting a protocol you ran a handful of times. Raise `height=` for a taller box.

The second cell is `ra.browse_experiment_tree()` — a dropdown of every MEA date. Pick one and it renders that date's blocks as a **date → protocol → datafile** tree, indexed on those three levels so pandas blanks the repeated labels and the nesting shows: each date once, each protocol once beneath it, and the datafiles that ran it underneath. Columns are the group label, the filter-wheel reading, the sorting chunk and the block duration. `ra.experiment_tree('20231220C')` returns the same table directly when you already know the date and don't want the widget.

Protocol names are shortened to their last dotted component — `edu.washington.riekelab.turner.protocols.EyeMovementTrajectoryAlternatingBackground` becomes `EyeMovementTrajectoryAlternatingBackground`. The prefix only records which lab package the protocol came from and makes every table unreadable.

**On the filter wheel.** `filter_wheel_ndf` is the epoch parameter `NDF`. In this codebase those are the same thing — `populate_ndf_column` calls it "the filter wheel ND being used" and the single-cell code renames the identical field to `filter_wheel_ndf`, so the tree uses the explicit name. It is read from the first epoch of each block via a JSON extraction pushed into SQL, which is why a date loads in one query instead of one per block.

In [ ]:
# Whole-database survey. Each helper is one bulk query joined in pandas —
# implementations live in retinanalysis/utils/db_summary.py.
#
# recording_summary is species x rig with totals on both margins. The old
# recording_counts table was exactly this one's column sums, so showing both
# was the same numbers twice.
display(ra.recording_summary())

protocol_counts = ra.mea_protocol_counts()
print(f'{len(protocol_counts)} distinct protocols across MEA dates, '
      f'ranked by number of dates:')

# Every protocol, not a head(15) truncation: scroll_table caps the box height
# and pins the header row, so the list only scrolls once it outgrows the box.
ra.scroll_table(protocol_counts.reset_index(), height = 420,
                num_cols = ('n_dates', 'n_blocks'));

In [4]:
# Dropdown of every MEA date; picking one loads that date's block tree.
# Without the GUI, for a date you already know:  ra.experiment_tree('20231220C')
ra.browse_experiment_tree();

## 4. Find datasets that ran the protocol

**This is the one cell to change when analyzing a different protocol.** `PROTOCOL_SEARCH` is a lowercase substring match against the protocol names in the database, so a fragment is enough — `'AlternatingBackground'` picks out the full Java class name `edu.washington.riekelab.turner.protocols.EyeMovementTrajectoryAlternatingBackground` without you typing it. Keep the fragment specific: searching `'eyemovement'` instead would also catch five unrelated protocols. Pick one off the §3 table.

The result is one row per epoch block, so a date that ran the protocol twice appears twice. Everything downstream indexes into this table by row, so its row numbers are the currency of the rest of the notebook.

**The cell says what it dropped, and why.** Two filters run over the raw search, and both used to remove rows silently, which is how you end up staring at five rows wondering where the rest went:

1. Dates with no Vision analysis directory on any *currently mounted* volume are removed, and named in the printout. They ran the protocol; their analysis output just isn't reachable right now, which is a statement about what's plugged in rather than about the data.
2. `ss_version` reports the Kilosort version each chunk actually has — `kilosort2.5` where present, otherwise `kilosort2`. **`not found` means that chunk is on no configured volume**, and the count of loadable chunks is printed alongside. The date-level filter above can't catch this, because a date directory can exist while the specific chunk inside it does not.

Read the `ss_version` column before picking anything in §5: a `not found` row has nothing to render.

The `experiment_id` / `protocol_id` / `group_id` / `block_id` / `chunk_id` columns are still on `exp_search` for anything downstream that needs them — the display is just narrowed to `SEARCH_COLS`. Widen that list, or `display(exp_search)`, to see everything.

In [ ]:
PROTOCOL_SEARCH = 'variableMeanDriftingGrating'   # <-- EDIT ME

exp_search = ra.get_datasets_from_protocol_names(PROTOCOL_SEARCH)
print(f'\nsearch found {len(exp_search)} blocks across '
      f'{exp_search["exp_name"].nunique()} dates')

# Keep only dates whose Vision analysis output is on a mounted volume. tier_dirs
# returns every configured analysis root, so this spans both SSDs and the NAS
# rather than only checking the top-priority one.
available_experiments = sorted({exp for root in ra.tier_dirs('analysis')
                                for exp in os.listdir(root)})

# Say what the filter removed. Dropping rows silently is how "why are there
# only five?" happens — these dates ran the protocol, their analysis output
# just isn't on any volume mounted right now.
missing = sorted(set(exp_search.query('exp_name not in @available_experiments')
                     ['exp_name']))
if missing:
    print(f'dropped {len(missing)} date(s) with no analysis directory on a '
          f'mounted volume: {", ".join(missing)}')

exp_search = exp_search.query('exp_name in @available_experiments').reset_index(drop = True)

# Kilosort version actually on disk for each chunk. 'not found' means the chunk
# is on no configured volume, so §5 cannot render it — the date directory
# existing is not enough.
exp_search = ra.add_ss_version_column(exp_search)
loadable = (exp_search['ss_version'] != 'not found').sum()
print(f'{len(exp_search)} blocks remain across '
      f'{exp_search["exp_name"].nunique()} dates; {loadable} have a loadable chunk\n')

# The experiment_id / protocol_id / group_id / block_id / chunk_id columns stay
# on exp_search for downstream code; they are just noise on screen.
SEARCH_COLS = ['exp_name', 'datafile_name', 'chunk_name', 'ss_version',
               'NDF', 'group_label']
display(exp_search[SEARCH_COLS])

## 5. Plot mosaics for candidate datasets

RF mosaics for the datasets you name, so you can judge which experiment has the cleanest typing before committing to one. Set `MOSAIC_ENTRIES` to row indices from the §4 table — pick rows whose `ss_version` isn't `not found`, since those have no chunk on any volume. Those same row indices are what §6 takes to build the pipeline, so whichever mosaic wins here you carry forward by its number.

The second cell says what is actually *in* each mosaic, which the pictures alone won't tell you. It's a **dropdown, not a loop**: `ra.browse_chunk_summaries(chunks)` shows one chunk at a time, so the cell's output stays the same length whether you loaded three datasets or fifteen. Each chunk renders on first selection and is then cached as an image, so revisiting one is instant and a chunk you never open costs nothing. The dropdown label carries the sort version and how many cells cleared `minimum_n` — usually enough to skip the thin ones without opening them.

For the selected chunk you get a table from `cell_type_summary`:

- **Cells per type.** A mosaic can look tidy while resting on four cells. It also lists the types you didn't ask for (`Unknown`, `OffMystery`, …), which is useful context — on `20231220C/chunk4` the largest single group is 286 `Unknown` cells against 116 `OffP`.
- **Firing rate per type**, as mean/median/min/max. This is the mean rate across the whole noise chunk, so it's a data-quality number, not a response measure. A type whose rates sit near zero is usually a sorting artifact rather than a population.

Then the three views a spatial mosaic can't give you, one column per cell type in `MAIN_TYPES` order so the columns line up with the mosaic above:

- **Temporal RF** — the green-channel STA time course, every cell in a dim line with the mean ± SEM over the top, on a millisecond axis ending at the spike. This is where a mislabeled type shows itself: an `OnP` group whose mean filter is flat, or inverts, was never a population no matter how regular its mosaic looks. Green is the channel plotted because the noise here is achromatic — red and green are identical — and one trace per type keeps the four columns comparable. The frame interval comes from the chunk's recorded `refreshPeriod` rather than an assumed 60 Hz, because STA depth varies between sorts (30 and 61 frames both occur).
- **Autocorrelation (ISI)** — Vision's autocorrelation histogram, sum-normalized per cell so a fast-firing cell doesn't dominate the mean. Read the left edge first: density at lags under ~2 ms is a refractory-period violation, which means the cluster merges more than one unit. A clean type dips to zero at zero lag and peaks at that type's preferred interval.
- **Spike count over the noise run** — one ECDF across all types, total spikes per cell. This is the "how much data is behind each STA" view, and it's the number that matters for whether an RF fit means anything; the rate columns in the table above are the same quantity divided by chunk duration, which is what you want when comparing chunks of different length instead.

The count distribution is drawn as an **ECDF rather than a histogram**, because a well-populated type here has ~100 cells and a marginal one has three, and at those counts a histogram's shape is mostly an artifact of where the bins fell. An ECDF is exact at any n — one step per cell — so a sparse type overlaid on a dense one stays honest, and curves separate vertically instead of occluding each other. Read it as: further right means more spikes, steeper means the type is more tightly clustered. The x axis is logarithmic because counts run over orders of magnitude between types; pass `log_x=False` for a linear one.

To render every chunk inline instead — for exporting the notebook, say, where a dropdown is dead — loop `ra.plot_chunk_panels(chunk, ...)` over `chunks.items()`. `ra.plot_spike_count_distribution` and `ra.plot_firing_rate_distribution` draw the bottom panel on its own.

**The noise chunk is picked by recording time, preferring one recorded before the protocol.** A chunk that ran afterwards has an intervening protocol's worth of adaptation between it and the data being typed, so it's used only when no preceding chunk works — even if the clock distance is shorter. Candidates lacking a typing file are skipped, so `20220823C` still falls through to a following chunk (only `chunk5` has an analysis dir at all); the printout says "before" or "after" so you can see when that happened. §6 pins the pipeline to whichever chunk was loaded here rather than deriving the choice a second time, so the mosaic you judge and the chunk the analysis types against are the same one.

Sort versions are resolved per chunk, and the analysis directory is looked up with `find_path`, which walks the volumes in read-priority order — so a chunk that only exists on the NAS is found there when ChrisProSSD doesn't have it. A chunk missing everywhere is reported and skipped rather than aborting the sweep.

In [ ]:
# <-- EDIT ME: row indices from the §4 table. Pick rows whose ss_version is not
# 'not found'; each costs one Vision analysis-chunk read.
MOSAIC_ENTRIES = [0, 1, 2, 3, 4]

mosaic_search = exp_search.loc[MOSAIC_ENTRIES]
display(mosaic_search[SEARCH_COLS])

# include_neurons pulls the spike times too, which the next cell needs for
# firing rates; return_chunks hands back the loaded chunks so that summary
# doesn't have to read them a second time.
all_axes, chunks = ra.plot_mosaics_for_datasets(
    mosaic_search, cell_types, minimum_n = 3, b_zoom = True,
    include_neurons = True, return_chunks = True)

In [ ]:
# What is actually in each mosaic: cells per type, temporal RF, autocorrelation
# and spike counts. One chunk at a time from the dropdown, so the output stays
# the same length however many entries MOSAIC_ENTRIES has.
ra.browse_chunk_summaries(chunks, cell_types = MAIN_TYPES, minimum_n = 3);

## 6. Initialize the analysis pipeline

Pick one of the datasets you just looked at and build the three objects the rest of the notebook runs on: the stimulus block, the response block (spike times from the Kilosort output), and the noise `AnalysisChunk` (STAs and EIs). It then cluster-matches the protocol's cells against that noise chunk so noise-derived cell types carry over to the protocol datafile.

**Name the dataset by row index, not by string.** `ENTRY` is a row index into the §4 table — the same indexing `MOSAIC_ENTRIES` uses in §5, so picking the mosaic you liked is a matter of copying its number across. `exp_name` and `datafile_name` are read off that row rather than retyped, which removes the failure mode the old two-string version had: several dates ran this protocol more than once (`20231220C` has both `data022` and `data023`, `20221101C` has three), and naming only the date silently matched the wrong block or raised on `.item()`.

**The noise chunk carries over from §5 rather than being re-derived.** If `ENTRY` was one of the rows you mosaicked, the pipeline is pinned to the chunk that browser actually loaded, so the cells it types against are the ones whose mosaic, temporal RFs and autocorrelations you just judged. Two independent implementations of "nearest chunk, preferring one recorded before the protocol" — the one §5 uses and the one `MEAStimBlock` uses — should agree, but pinning means you never have to wonder. For a row you didn't mosaic, `chunk_name` stays `None` and `create_mea_pipeline` picks by that same rule, skipping candidates without a typing file.

**Nothing re-resolves the Kilosort version.** §4 already resolved it per chunk and the answer is in the `ss_version` column, printed here for the record. Note it is deliberately *not* passed to `create_mea_pipeline`: that function's `ss_version` argument is a single blanket override applied to both trees, while the sorted-data tree (for the response block) and the analysis tree (for the chunk) live on different volumes and can legitimately carry different sorts. Left alone, each is detected independently — `kilosort2.5` when present, `kilosort2` otherwise. Pass `ss_version=...` only when you want to pin both to one specific sort.

Two things in the printout are worth reading: how far away the chosen noise chunk was recorded, and the cluster-match rate. A chunk recorded hours from the protocol matches worse, and a low match rate is the usual explanation for a cell type looking scattered in §7.

In [ ]:
ENTRY = 4   # <-- EDIT ME: row index from the §4 table, same indexing as MOSAIC_ENTRIES

entry         = exp_search.loc[ENTRY]
exp_name      = entry['exp_name']
datafile_name = entry['datafile_name']

# Type against the chunk §5 actually loaded for this date, so the pipeline uses
# the same mosaic you just judged.
#
# §5 is optional, so read `chunks` off the namespace rather than assuming it:
# once you know which row you want you go straight from §4 to here, and a bare
# reference would NameError on a perfectly reasonable way to run the notebook.
# Without it — or for a row you didn't mosaic — chunk_name stays None and
# create_mea_pipeline picks by its own nearest-preceding-chunk rule.
mosaicked = globals().get('chunks', {})
loaded_chunks = sorted({key.split('/', 1)[1] for key in mosaicked
                        if key.startswith(f'{exp_name}/')})
chunk_name = loaded_chunks[0] if len(loaded_chunks) == 1 else None

print(f'{exp_name} / {datafile_name}  ({entry["ss_version"]} per §4)')
print(f'noise chunk: {chunk_name or "not mosaicked — letting the pipeline choose"}\n')

# ss_version is deliberately not passed: it is one blanket override for both the
# sorted-data and analysis trees, which can carry different sorts. Left off, each
# is detected independently.
pipeline = ra.create_mea_pipeline(exp_name, datafile_name,
                                  analysis_chunk_name = chunk_name)

## 7. Inspect the dataset

Every check on the dataset §6 built, in one tabbed view. This used to be four sections you scrolled through in order, which meant that changing `ENTRY` in §6 sent you back through all of them — and the checks are the part you actually repeat, because the usual outcome of looking is picking a different dataset.

Tabs render on first look and are kept after, so the expensive ones cost nothing until you open them: the cluster-match tab re-runs an EI correlation pass, and a dense raster is a few hundred thousand ticks to draw.

**Cluster match** — do the noise chunk and the protocol datafile describe the same cells? The distribution of each noise cell's best EI correlation, split by whether it matched, with the cutoff marked; then one example per outcome, matched pairs across the accepted range and one instance of each rejection mode. Read the rejection tally before treating the match rate as a data-quality number: `claimed_by_other` at r ≈ 0.99 means split or duplicated units, not bad data, and it is usually the largest category.

**RFs** — do the matched cells still form the mosaic they were typed from? Protocol-side RFs above, noise-chunk RFs below, **on identical axes**. That matters: left to themselves each figure zooms to its own cells, and since the protocol mosaic is drawn from the subset that survived matching, it lands on a tighter window and the same population appears to have spread out. The union of both windows is applied to every axis in both figures, so a difference you can see is a difference in the data. A type well-formed in the noise mosaic and scattered in the protocol one is a matching failure — go back to the first tab.

**Rasters** — the raw spikes, first epochs on top and last below, y is cell ID, one panel per epoch. Cell order is identical in every panel so a row reads straight across.

**Spikes/epoch** — the same thing counted. The summary is one line per cell type, absolute above and relative to each type's own mean below; the heatmap under it is the cell × epoch distribution those means average away.

Each tab is a public function if you want it on its own: `ra.plot_match_qc`, `pipeline.plot_rf_comparison`, `ra.browse_epoch_rasters`, `ra.plot_epoch_spike_counts` and `ra.browse_epoch_count_heatmaps`. Use those when exporting the notebook, where tabs and dropdowns are dead.

In [ ]:
# Cluster match, RFs, rasters and per-epoch counts, one tab each. Tabs build
# on first look, so re-running this after changing ENTRY in §6 is cheap until
# you actually open a tab.
pipeline.inspect(cell_types = MAIN_TYPES, minimum_n = 3);

## 8. Break out the objects — and hand off

The end of the shared part. `MEAPipeline` is a thin container over three members; pulling them into their own names is what a per-protocol notebook starts from.

- `stim_block` — `df_epochs` (one row per epoch, with the protocol's parameters promoted to columns where they vary) and `d_epoch_block_params` (what was fixed for the whole block). **This is where protocol-specific analysis begins**, because this is the first object whose contents differ between protocols.
- `response_block` — `df_spike_times`, one row per cell, each holding a list of per-epoch spike-time arrays in ms, with `cell_type` and `noise_id` filled in from the cluster match.
- `analysis_chunk` — the noise-derived side: RF parameters, STAs, EIs, timecourses, autocorrelations.

**To analyze a new protocol, copy this notebook**, change the search string in §4, and continue past here with that protocol's condition axes — group epochs by the parameters that varied, average PSTHs within condition, whatever the experiment was asking. Everything above stays as written and stays shared, so a fix to the pipeline or the QC reaches every protocol at once.

Useful next steps that are already written: `ra.get_spike_xarr(response_block, cell_types=...)` for a ragged (cell × epoch) array of spike times, `ra.plot_raster_with_psth` for a raster and PSTH per type, and `ra.epoch_count_matrix(response_block, cell_type)` for the (cells × epochs) count matrix behind §7's heatmap.

In [10]:
stim_block = pipeline.stim
response_block = pipeline.resp
analysis_chunk = pipeline.analysis_chunk